# Neural Network in NumPy — Predicting Exam Success
This notebook is my own version of the "Simple Neural Network in NumPy" lab from Andrew Ng's *Advanced Learning Algorithms* course (Course 2, Week 1). The original lab used a **coffee roasting** dataset (Temperature vs. Duration) to predict whether a roast is good. Here I use a completely different dataset — **Hours Studied vs. Hours Slept**, predicting whether a student **passes an exam** — but the underlying concept is exactly the same:

- Build a 2-layer neural network **from scratch using only NumPy** (no `model.fit`, no autograd).
- Implement a `dense` layer function and chain two of them together (`sequential`).
- Verify that the hand-written NumPy forward pass produces the **same predictions** as a trained TensorFlow model.
- Visualize the learned decision boundary.

Just like the coffee roasting problem, "passing the exam" isn't as simple as "study more = pass". There's a trade-off: study too little and you fail, but study so much that you barely sleep and you *also* fail. That non-linear, "good region surrounded by bad regions" shape is exactly why we need a neural network instead of a single straight-line decision boundary.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import logging
logging.getLogger("tensorflow").setLevel(logging.ERROR)
tf.autograph.set_verbosity(0)

np.set_printoptions(precision=4)
plt.rcParams['figure.figsize'] = (6, 4)


def sigmoid(z):
    """Numerically stable sigmoid activation function."""
    return 1 / (1 + np.exp(-z))


## 1. Dataset

The two features are:
- **Hours Studied** (0–10)
- **Hours Slept** (4–10, the night before the exam)

A student **passes (`y=1`)** only if they studied a *moderate* amount **and** got enough sleep. Studying a lot is only good up to a point — cramming all night at the cost of sleep still leads to failure. This mirrors the coffee roasting lab, where a good roast needed the *right combination* of temperature and duration, not just "more of both".

In [ ]:
def load_study_data(n=250, seed=3):
    """Generates a synthetic exam-performance dataset.

    Returns:
        X (ndarray (n,2)): columns = [hours_studied, hours_slept]
        Y (ndarray (n,1)): 1 = pass, 0 = fail
    """
    rng = np.random.default_rng(seed)
    study = rng.uniform(0, 10, n)
    sleep = rng.uniform(4, 10, n)
    X = np.stack([study, sleep], axis=1)

    # The "good" (passing) zone is a diagonal band: the more you study,
    # the less extra sleep you need to have already banked, but you can
    # never skip sleep entirely and you can't pass by studying 0 hours.
    lower_bound = 8 - 0.35 * study
    upper_bound = 10 - 0.35 * study
    passed = (sleep >= lower_bound) & (sleep <= upper_bound) & (study >= 2) & (study <= 9)

    Y = passed.astype(float).reshape(-1, 1)
    return X, Y


X, Y = load_study_data()
print(f"X shape: {X.shape}, Y shape: {Y.shape}")
print(f"Number who passed: {int(Y.sum())} / {len(Y)}")


Let's plot the data below. Each point is one student: green circles passed, red x's failed.

In [ ]:
def plt_data(X, Y, ax):
    pos = Y[:, 0] == 1
    neg = Y[:, 0] == 0
    ax.scatter(X[pos, 0], X[pos, 1], marker='o', c='green', edgecolors='k', label='Pass (y=1)')
    ax.scatter(X[neg, 0], X[neg, 1], marker='x', c='red', label='Fail (y=0)')
    ax.set_xlabel('Hours Studied')
    ax.set_ylabel('Hours Slept')


fig, ax = plt.subplots(1, 1)
plt_data(X, Y, ax)
ax.set_title("Exam Outcome vs. Study/Sleep Hours")
ax.legend()
plt.show()


### Normalize the data
Just like in the coffee roasting lab, we normalize the features so both are on a similar scale (zero mean, unit variance) before training. This helps the network train faster and more reliably.

In [ ]:
print(f"Study Max, Min pre normalization: {np.max(X[:,0]):0.2f}, {np.min(X[:,0]):0.2f}")
print(f"Sleep Max, Min pre normalization: {np.max(X[:,1]):0.2f}, {np.min(X[:,1]):0.2f}")

norm_l = tf.keras.layers.Normalization(axis=-1)
norm_l.adapt(X)  # learns mean, variance from the training data
Xn = norm_l(X)

print(f"Study Max, Min post normalization: {np.max(Xn[:,0]):0.2f}, {np.min(Xn[:,0]):0.2f}")
print(f"Sleep Max, Min post normalization: {np.max(Xn[:,1]):0.2f}, {np.min(Xn[:,1]):0.2f}")


## 2. Train a small model in TensorFlow

In the original lab, trained weights were simply copied over from a previous notebook. Since this is a brand new dataset, we train a tiny model here ourselves: two Dense layers with sigmoid activations (3 units, then 1 unit) — the same architecture as the coffee roasting network. Once trained, we'll pull out the raw weight matrices and reimplement the forward pass by hand in NumPy.

In [ ]:
tf.random.set_seed(1234)

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(2,)),
    tf.keras.layers.Dense(3, activation='sigmoid', name='layer1'),
    tf.keras.layers.Dense(1, activation='sigmoid', name='layer2'),
])

model.compile(
    loss=tf.keras.losses.BinaryCrossentropy(),
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.05),
)

history = model.fit(Xn, Y, epochs=200, verbose=0)

train_preds = model.predict(Xn, verbose=0)
train_acc = np.mean((train_preds >= 0.5).astype(int) == Y)
print(f"Final training loss: {history.history['loss'][-1]:.4f}")
print(f"Training accuracy:   {train_acc*100:.1f}%")


In [ ]:
W1_tmp, b1_tmp = model.get_layer('layer1').get_weights()
W2_tmp, b2_tmp = model.get_layer('layer2').get_weights()

print("W1 shape:", W1_tmp.shape, " b1 shape:", b1_tmp.shape)
print("W2 shape:", W2_tmp.shape, " b2 shape:", b2_tmp.shape)


## 3. NumPy Model (Forward Prop in NumPy)

Now for the main point of the lab: reimplementing the forward pass **without TensorFlow**, using nothing but NumPy arrays and a for-loop.

A layer contains multiple units. For each unit `j` in the layer, we take the dot product of that unit's weight column `W[:,j]` with the input, add the bias `b[j]`, to get `z`, then apply the activation function `g(z)`. Stacking two of these dense layers gives us a full 2-layer network.

In [ ]:
# Activation function for both layers
g = sigmoid


In [ ]:
def my_dense(a_in, W, b):
    """
    Computes a dense layer forward pass.
    Args:
      a_in (ndarray (n,))  : Data, 1 example, n features
      W    (ndarray (n,j)) : Weight matrix, n features per unit, j units
      b    (ndarray (j,))  : bias vector, j units
    Returns:
      a_out (ndarray (j,)) : activations of the j units
    """
    units = W.shape[1]
    a_out = np.zeros(units)
    for j in range(units):
        w = W[:, j]
        z = np.dot(w, a_in) + b[j]
        a_out[j] = g(z)
    return a_out


*Note: `g` is hard-coded as sigmoid here since both layers use it in this lab. You could also pass `g` in as a parameter to make `my_dense` reusable with other activations.*

The cell below chains two `my_dense` layers together to form a full 2-layer network:

In [ ]:
def my_sequential(x, W1, b1, W2, b2):
    a1 = my_dense(x,  W1, b1)
    a2 = my_dense(a1, W2, b2)
    return a2


We'll use the weights `W1_tmp, b1_tmp, W2_tmp, b2_tmp` extracted from the TensorFlow model we trained above — these are the "trained weights" for our NumPy network.

### Predictions

Once we have trained weights, we can make predictions. The output of the model is a **probability** (of passing the exam). To turn that into a pass/fail decision, we apply a threshold — here, 0.5.

First, a routine similar to TensorFlow's `model.predict()`, that runs every row of a matrix `X` through the network:

In [ ]:
def my_predict(X, W1, b1, W2, b2):
    m = X.shape[0]
    p = np.zeros((m, 1))
    for i in range(m):
        p[i, 0] = my_sequential(X[i], W1, b1, W2, b2)[0]
    return p


Let's try it on two hand-picked examples:

In [ ]:
X_tst = np.array([
    [6, 7.5],   # studied a reasonable amount and slept well -> expect PASS
    [9, 4.5]])  # crammed all night, barely slept -> expect FAIL
X_tstn = norm_l(X_tst)  # remember to normalize using the same norm_l layer
predictions = my_predict(X_tstn, W1_tmp, b1_tmp, W2_tmp, b2_tmp)
print(f"predicted probabilities = \n{predictions}")


To convert probabilities into decisions, apply the threshold:

In [ ]:
yhat = np.zeros_like(predictions)
for i in range(len(predictions)):
    if predictions[i] >= 0.5:
        yhat[i] = 1
    else:
        yhat[i] = 0
print(f"decisions = \n{yhat}")


This can be written more succinctly using vectorized NumPy operations:

In [ ]:
yhat = (predictions >= 0.5).astype(int)
print(f"decisions = \n{yhat}")


## 4. Sanity check: does the NumPy model match TensorFlow?

Since we hand-copied the weights, our NumPy forward pass should produce (almost) identical outputs to the original TensorFlow model.

In [ ]:
tf_preds_on_test = model.predict(X_tstn, verbose=0)
print("TensorFlow predictions: \n", tf_preds_on_test)
print("NumPy predictions:      \n", predictions)
print("\nMatch within tolerance:", np.allclose(predictions, tf_preds_on_test, atol=1e-5))


## 5. Network function — visualizing the decision boundary

The plot below shows the operation of the whole network across the full input space.
- **Left**: the raw probability output of the network (green = high probability of passing), overlaid on the training data.
- **Right**: the same output after applying the 0.5 threshold, so it becomes a hard pass/fail region.

Notice the diagonal "good" band — a single straight-line boundary could never capture this shape, but our small 2-layer network learns it easily, just like the coffee roasting network learned its own diagonal "good roast" region.

In [ ]:
def plt_network(X, Y, predict_fn, study_range=(0, 10), sleep_range=(4, 10), resolution=80):
    study_vals = np.linspace(*study_range, resolution)
    sleep_vals = np.linspace(*sleep_range, resolution)
    SS, LL = np.meshgrid(study_vals, sleep_vals)
    grid = np.stack([SS.ravel(), LL.ravel()], axis=1)
    grid_n = np.array(norm_l(grid))
    grid_p = predict_fn(grid_n).reshape(SS.shape)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))

    c0 = axes[0].pcolormesh(SS, LL, grid_p, cmap='RdYlGn', shading='auto', vmin=0, vmax=1)
    plt_data(X, Y, axes[0])
    axes[0].set_title("Raw probability output")
    fig.colorbar(c0, ax=axes[0], label='P(pass)')

    c1 = axes[1].pcolormesh(SS, LL, (grid_p >= 0.5).astype(int), cmap='RdYlGn', shading='auto', vmin=0, vmax=1)
    plt_data(X, Y, axes[1])
    axes[1].set_title("Thresholded decision (0.5)")

    plt.tight_layout()
    plt.show()


netf = lambda x: my_predict(x, W1_tmp, b1_tmp, W2_tmp, b2_tmp)
plt_network(X, Y, netf)


## Congratulations!

You've built a small neural network in NumPy — from raw weight matrices to a full forward pass to a visualized decision boundary — on a completely different dataset than the original lab. The takeaway is the same one from the coffee roasting lab: a "dense layer" is really just a handful of dot products and an activation function, looped over units and then over layers. Everything more advanced (TensorFlow, PyTorch, etc.) is built on exactly this idea, just optimized and automated.